# Google Earth Engine Feature Extractor - Test Notebook
Testing the 7-feature vegetation extractor for wildfire spread prediction

**Features Extracted:**
1. VIIRS band M11 (thermal)
2. VIIRS band I2 (near-IR)
3. NDVI (vegetation greenness)
4. EVI2 (enhanced vegetation index)
5. Total precipitation (30-day cumulative)
6. Wind speed (monthly average)
7. Elevation (static topographic)

In [ ]:
# Import required libraries
import sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings

# Add featureAPI to path
sys.path.insert(0, '/u50/joshin10/WildfireSpreadPrediction/featureAPI')

from gee_feature_extractor import GEEFeatureExtractor, normalize_features

print("Imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

## Initialize GEE Feature Extractor

First, authenticate with Google Earth Engine:
```bash
earthengine authenticate
```

Then initialize the extractor in the notebook.

In [ ]:
# Initialize the feature extractor
# This assumes you've already run: earthengine authenticate
extractor = GEEFeatureExtractor()
print("GEE Feature Extractor initialized successfully!")
print(f"\nFeatures to extract ({len(extractor.FEATURE_NAMES)}):")
for i, name in enumerate(extractor.FEATURE_NAMES):
    print(f"  {i}: {name}")

## Extract Features for a Test Location

Test with real fire location coordinates. This will extract features for a 1km x 1km region at 250m resolution (4x4 pixels).

In [ ]:
# Test location: Northern California (historically active wildfire region)
test_latitude = 39.5  # Near Paradise fire region
test_longitude = -121.6
test_date = "2023-08-15"  # Mid-summer, fire season

print(f"Extracting features for:")
print(f"  Location: ({test_latitude:.2f}°N, {test_longitude:.2f}°E)")
print(f"  Date: {test_date}")
print(f"  Region size: 1 km x 1 km")
print(f"  Resolution: 250m (4x4 pixels)")
print()

# Extract features
# NOTE: This will fail if you haven't authenticated with GEE
# To authenticate: earthengine authenticate
try:
    features, feature_names = extractor.extract_features(
        latitude=test_latitude,
        longitude=test_longitude,
        date=test_date,
        region_size_meters=1000,
        resolution_meters=250
    )
    
    print(f"✓ Extraction successful!")
    print(f"  Shape: {features.shape} (7 features, {features.shape[1]}x{features.shape[2]} pixels)")
    print()
    print("Feature statistics:")
    for i, name in enumerate(feature_names):
        feat = features[i]
        print(f"  {name}:")
        print(f"    Min: {np.nanmin(feat):.4f}, Max: {np.nanmax(feat):.4f}, Mean: {np.nanmean(feat):.4f}")
    
except Exception as e:
    print(f"✗ Extraction failed: {e}")
    print("\nMake sure you've authenticated with GEE:")
    print("  earthengine authenticate")
    # Create dummy data for testing visualization
    features = np.random.randn(7, 4, 4) * 0.5 + 0.2
    features = np.clip(features, 0, 1)
    print("\nUsing random dummy data for visualization...")

## Visualize Extracted Features

Display each of the 7 features as heatmaps to understand their spatial distribution.

In [ ]:
# Visualize all 7 features
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, (feat, name) in enumerate(zip(features, extractor.FEATURE_NAMES)):
    ax = axes[i]
    im = ax.imshow(feat, cmap='viridis', aspect='auto')
    ax.set_title(f'{i}: {name}', fontsize=10, fontweight='bold')
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')
    plt.colorbar(im, ax=ax, label='Value')

# Hide the last empty subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.suptitle('7 Significant Vegetation Features for Wildfire Prediction\n(1km x 1km region at 250m resolution)', 
             fontsize=12, fontweight='bold', y=1.00)
plt.show()

print("Visualization complete!")

## Normalize Features for Model Input

The extracted features need to be normalized to match the distribution the model was trained on.

In [ ]:
# Normalize features using default standardization
normalized_features = normalize_features(features)

print("Feature normalization:")
print(f"  Original shape: {features.shape}")
print(f"  Normalized shape: {normalized_features.shape}")
print()
print("Normalized feature statistics:")
for i, name in enumerate(extractor.FEATURE_NAMES):
    feat = normalized_features[i]
    print(f"  {name}:")
    print(f"    Min: {np.nanmin(feat):.4f}, Max: {np.nanmax(feat):.4f}, Mean: {np.nanmean(feat):.4f}")
print()
print("✓ Features are now ready to feed into the Domain-Adversarial UTAE model!")
print(f"\nInput shape for model: (1, {normalized_features.shape[0]}, {normalized_features.shape[1]}, {normalized_features.shape[2]})")
print("  = (batch_size=1, channels=7, height=4, width=4)")

## Next Steps: Model Inference

To use these features for prediction with the Domain-Adversarial UTAE model:

```python
import torch
import sys
sys.path.insert(0, '../src/wsts')

from models import DomainAdversarialUTAELightning

# Load trained model
model = DomainAdversarialUTAELightning.load_from_checkpoint('path/to/checkpoint.ckpt')
model.eval()

# Convert normalized features to torch tensor and add temporal dimension
# Shape: (1, T=1, C=7, H, W) - single timestep
input_tensor = torch.from_numpy(normalized_features).unsqueeze(0).unsqueeze(1).float()

# Predict fire spread
with torch.no_grad():
    fire_prediction = model(input_tensor)  # Shape: (1, 1, H, W)

# Extract probability and threshold
fire_prob = torch.sigmoid(fire_prediction).numpy()
fire_mask = (fire_prob > 0.5).astype(int)

print(f"Prediction shape: {fire_prob.shape}")
print(f"Fire probability range: [{fire_prob.min():.4f}, {fire_prob.max():.4f}]")
print(f"Pixels with >50% fire risk: {(fire_mask > 0).sum()}")
```